In [2]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot"

os.environ["HADOOP_HOME"] = r"C:\hadoop"

os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print(
    "winutils exists =",
    os.path.exists(r"C:\hadoop\bin\winutils.exe"),
)
print(
    "hadoop.dll exists =",
    os.path.exists(r"C:\hadoop\bin\hadoop.dll"),
)

JAVA_HOME = C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot
HADOOP_HOME = C:\hadoop
winutils exists = True
hadoop.dll exists = True


In [3]:
from pathlib import Path
import os

if "project_path" not in globals():
    project_path = Path.cwd().parent
    os.chdir(project_path)

print("Project path:", project_path)

Project path: c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


In [4]:
%load_ext autoreload
%autoreload 2
from __future__ import annotations

from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

from credit_risk.pipelines.data_preprocess import (
    create_spark_session,
    create_path,
    read_spark_parquet,
    build_master_dataset_spark,
    build_performance_spark,
)

from credit_risk.features.origination_spark import (
    build_origination_spark,
)

from credit_risk.features.behavioral_spark import (
    add_calculated_loan_age_spark,
    add_prior_serious_delinquency_flag_spark,
    build_behavioral_features_spark,
)

from credit_risk.target.behavioral_spark import (
    build_behavioral_target_spark,
)
from credit_risk.utils.config import read_config

In [5]:
config = read_config(project_path)

In [6]:
os.getcwd()

'c:\\Users\\vorad\\OneDrive\\Desktop\\Projects\\mortgage-credit-risk'

In [6]:
def add_calculated_loan_age_spark(
    df: DataFrame,
) -> DataFrame:
    """
    Calculate loan age from the canonical monthly Period[M] ordinal.

    Pandas reads the Parquet fields as Period[M], e.g.:

        2015-05

    Spark reads the underlying monthly ordinal, e.g.:

        544

    The Pandas calculation:

        months(period - first_payment_date) + 1

    is therefore equivalent to:

        period - first_payment_date + 1
    """

    # _validate_required_columns(
    #     df,
    #     {
    #         "period",
    #         "first_payment_date",
    #     },
    #     "calculated loan age",
    # )

    return df.withColumn(
        "calculated_loan_age",
        (F.col("period") - F.col("first_payment_date") + F.lit(1)).cast("int"),
    )

In [7]:
spark = create_spark_session(config)

origination_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "origination_path",
    "freddie_mac",
    2015,
)

performance_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "performance_path",
    "freddie_mac",
    2015,
)
origination_df = spark.read.parquet(str(os.getcwd()/ origination_path))

performance_df = spark.read.parquet(str(os.getcwd() / performance_path))

origination_features = build_origination_spark(
    origination_df,
    config,
)

origination_dates = origination_df.select(
    "loan_id",
    "first_payment_date",
).dropDuplicates(
    ["loan_id"],
)

origination_features = origination_features.join(
    origination_dates,
    on="loan_id",
    how="left",
)

performance_features = build_performance_spark(
    performance_df,
)

master = build_master_dataset_spark(
    origination_features,
    performance_features,
)

master = add_calculated_loan_age_spark(
    master,
)

c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [8]:
master.select(
    "period",
    "first_payment_date",
).show(20, truncate=False)

+------+------------------+
|period|first_payment_date|
+------+------------------+
|551   |552               |
|552   |552               |
|553   |552               |
|554   |552               |
|555   |552               |
|556   |552               |
|557   |552               |
|558   |552               |
|559   |552               |
|560   |552               |
|561   |552               |
|562   |552               |
|563   |552               |
|564   |552               |
|565   |552               |
|566   |552               |
|567   |552               |
|568   |552               |
|569   |552               |
|570   |552               |
+------+------------------+
only showing top 20 rows


In [9]:
ages = config["parameters"]["behavioral"]["observation_ages"]

print("Configured observation ages:")
print(ages)

print("\nCalculated loan age distribution:")

(
    master.groupBy("calculated_loan_age")
    .count()
    .orderBy("calculated_loan_age")
    .show(40, truncate=False)
)

print("\nRows at configured observation ages:")

(
    master.filter(F.col("calculated_loan_age").isin(ages))
    .groupBy("calculated_loan_age")
    .count()
    .orderBy("calculated_loan_age")
    .show()
)

Configured observation ages:
[6, 12]

Calculated loan age distribution:
+-------------------+-------+
|calculated_loan_age|count  |
+-------------------+-------+
|-1                 |18     |
|0                  |1292003|
|1                  |1394504|
|2                  |1421853|
|3                  |1424938|
|4                  |1422576|
|5                  |1418727|
|6                  |1412959|
|7                  |1405045|
|8                  |1395671|
|9                  |1385834|
|10                 |1375332|
|11                 |1364054|
|12                 |1351990|
|13                 |1338978|
|14                 |1325468|
|15                 |1312232|
|16                 |1299312|
|17                 |1286675|
|18                 |1274142|
|19                 |1261977|
|20                 |1250380|
|21                 |1239683|
|22                 |1229460|
|23                 |1219607|
|24                 |1209547|
|25                 |1198838|
|26                 |1188079

In [8]:
behavioral_features = build_behavioral_features_spark(
    master,
    config,
)

In [9]:
print(
    "Behavioral feature columns:",
    len(behavioral_features.columns),
)

print(
    "Behavioral feature rows:",
    behavioral_features.count(),
)

Behavioral feature columns: 44
Behavioral feature rows: 2741661


In [10]:
target = build_behavioral_target_spark(
    master,
    config,
)

In [11]:
(target.groupBy("observation_age").count().orderBy("observation_age").show())

+---------------+-------+
|observation_age|  count|
+---------------+-------+
|              6|1404307|
|             12|1338616|
+---------------+-------+



In [12]:
(target.groupBy("future_90dpd_12m").count().orderBy("future_90dpd_12m").show())

+----------------+-------+
|future_90dpd_12m|  count|
+----------------+-------+
|               0|2733840|
|               1|   9083|
+----------------+-------+



In [13]:
(
    behavioral_features.groupBy("observation_age")
    .count()
    .orderBy("observation_age")
    .show()
)

+---------------+-------+
|observation_age|  count|
+---------------+-------+
|              6|1404425|
|             12|1337236|
+---------------+-------+



In [14]:
modelling = behavioral_features.join(
    target,
    on=[
        "loan_id",
        "observation_age",
    ],
    how="inner",
)

print("Final modelling rows:", modelling.count())

(modelling.groupBy("observation_age").count().orderBy("observation_age").show())

Final modelling rows: 2741047
+---------------+-------+
|observation_age|  count|
+---------------+-------+
|              6|1403922|
|             12|1337125|
+---------------+-------+



In [15]:
(modelling.groupBy("future_90dpd_12m").count().orderBy("future_90dpd_12m").show())

+----------------+-------+
|future_90dpd_12m|  count|
+----------------+-------+
|               0|2733454|
|               1|   7593|
+----------------+-------+



In [16]:
# 1. Grain
duplicate_keys = (
    modelling.groupBy("loan_id", "observation_age").count().filter(F.col("count") > 1)
)

print("Duplicate keys:", duplicate_keys.count())

Duplicate keys: 0


In [17]:
# 2. Lifecycle consistency
invalid_age = modelling.filter(F.col("calculated_loan_age") != F.col("observation_age"))

print("Invalid calculated ages:", invalid_age.count())

Invalid calculated ages: 0


In [18]:
# 3. Final event rate
(
    modelling.groupBy("observation_age")
    .agg(
        F.count("*").alias("rows"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("observation_age")
    .show()
)

+---------------+-------+------+--------------------+
|observation_age|   rows|events|          event_rate|
+---------------+-------+------+--------------------+
|              6|1403922|  3105| 0.00221166133161244|
|             12|1337125|  4488|0.003356455080863...|
+---------------+-------+------+--------------------+



In [29]:
print("Rows at configured ages:")

ages = [6,12]
rows_at_age = master.filter(F.col("calculated_loan_age").isin(ages)).count()

print(rows_at_age)

print("\nRows after termination filter:")

rows_not_terminated = (
    master.filter(F.col("calculated_loan_age").isin(ages))
    .filter(F.col("zero_balance_code").isNull())
    .count()
)

print(rows_not_terminated)

Rows at configured ages:
2764949

Rows after termination filter:
2743537


In [10]:
sample = (
    master.select(
        "loan_id",
        "period",
        "first_payment_date",
    )
    .limit(30)
    .collect()
)

for row in sample:
    print(row)

Row(loan_id='F15Q40000001', period=551, first_payment_date=552)
Row(loan_id='F15Q40000001', period=552, first_payment_date=552)
Row(loan_id='F15Q40000001', period=553, first_payment_date=552)
Row(loan_id='F15Q40000001', period=554, first_payment_date=552)
Row(loan_id='F15Q40000001', period=555, first_payment_date=552)
Row(loan_id='F15Q40000001', period=556, first_payment_date=552)
Row(loan_id='F15Q40000001', period=557, first_payment_date=552)
Row(loan_id='F15Q40000001', period=558, first_payment_date=552)
Row(loan_id='F15Q40000001', period=559, first_payment_date=552)
Row(loan_id='F15Q40000001', period=560, first_payment_date=552)
Row(loan_id='F15Q40000001', period=561, first_payment_date=552)
Row(loan_id='F15Q40000001', period=562, first_payment_date=552)
Row(loan_id='F15Q40000001', period=563, first_payment_date=552)
Row(loan_id='F15Q40000001', period=564, first_payment_date=552)
Row(loan_id='F15Q40000001', period=565, first_payment_date=552)
Row(loan_id='F15Q40000001', period=566, 

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

origination_path = Path("data/02_intermediate/freddie_mac/2015/origination.parquet")

origination_raw = spark.read.parquet(str(origination_path))

print("Columns:", len(origination_raw.columns))
print(origination_raw.columns)

Columns: 31
['credit_score', 'first_payment_date', 'first_time_homebuyer_flag', 'maturity_date', 'msa', 'mi_percentage', 'number_of_units', 'occupancy_status', 'original_cltv', 'original_dti', 'original_upb', 'original_ltv', 'original_interest_rate', 'channel', 'prepayment_penalty_flag', 'amortization_type', 'property_state', 'property_type', 'postal_code', 'loan_id', 'loan_purpose', 'original_loan_term', 'number_of_borrowers', 'seller_name', 'super_conforming_flag', 'pre_harp_loan_id', 'special_eligibility_program', 'harp_indicator', 'property_valuation_method', 'interest_only_indicator', 'vantage_score_4']


In [12]:
origination_raw.printSchema()

root
 |-- credit_score: long (nullable = true)
 |-- first_payment_date: long (nullable = true)
 |-- first_time_homebuyer_flag: string (nullable = true)
 |-- maturity_date: long (nullable = true)
 |-- msa: string (nullable = true)
 |-- mi_percentage: long (nullable = true)
 |-- number_of_units: long (nullable = true)
 |-- occupancy_status: string (nullable = true)
 |-- original_cltv: long (nullable = true)
 |-- original_dti: long (nullable = true)
 |-- original_upb: long (nullable = true)
 |-- original_ltv: long (nullable = true)
 |-- original_interest_rate: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- prepayment_penalty_flag: string (nullable = true)
 |-- amortization_type: string (nullable = true)
 |-- property_state: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- loan_id: string (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- original_loan_term: long (nullable = true)
 |-

In [13]:
origination_raw.select(
    "first_payment_date",
    "maturity_date",
).show(
    30,
    truncate=False,
)

+------------------+-------------+
|first_payment_date|maturity_date|
+------------------+-------------+
|544               |723          |
|543               |902          |
|542               |721          |
|544               |903          |
|544               |903          |
|544               |903          |
|542               |901          |
|543               |902          |
|542               |901          |
|543               |722          |
|542               |901          |
|544               |723          |
|543               |902          |
|542               |721          |
|543               |902          |
|544               |903          |
|542               |721          |
|542               |901          |
|543               |902          |
|542               |721          |
|543               |902          |
|544               |903          |
|542               |901          |
|543               |722          |
|542               |901          |
|542               |

In [14]:
print(
    "first_payment_date:",
    origination_raw.schema["first_payment_date"].dataType,
)

print(
    "maturity_date:",
    origination_raw.schema["maturity_date"].dataType,
)

first_payment_date: LongType()
maturity_date: LongType()


In [16]:
import pandas as pd
from pathlib import Path

origination_path = Path("data/02_intermediate/freddie_mac/2015/origination.parquet")

orig_pd = pd.read_parquet(origination_path)

print("Shape:", orig_pd.shape)
print("\nDtypes:")
print(
    orig_pd[
        [
            "loan_id",
            "first_payment_date",
            "maturity_date",
        ]
    ].dtypes
)

print("\nSample:")
print(
    orig_pd[
        [
            "loan_id",
            "first_payment_date",
            "maturity_date",
        ]
    ]
    .head(20)
    .to_string(index=False)
)

Shape: (1474376, 31)

Dtypes:
loan_id                  string
first_payment_date    period[M]
maturity_date         period[M]
dtype: object

Sample:
     loan_id first_payment_date maturity_date
F15Q10000001            2015-05       2030-04
F15Q10000002            2015-04       2045-03
F15Q10000003            2015-03       2030-02
F15Q10000004            2015-05       2045-04
F15Q10000005            2015-05       2045-04
F15Q10000006            2015-05       2045-04
F15Q10000007            2015-03       2045-02
F15Q10000008            2015-04       2045-03
F15Q10000009            2015-03       2045-02
F15Q10000010            2015-04       2030-03
F15Q10000011            2015-03       2045-02
F15Q10000012            2015-05       2030-04
F15Q10000013            2015-04       2045-03
F15Q10000015            2015-03       2030-02
F15Q10000016            2015-04       2045-03
F15Q10000017            2015-05       2045-04
F15Q10000018            2015-03       2030-02
F15Q10000019           

In [7]:
spark = create_spark_session(config)
model_input = spark.read.parquet("C:/Users/vorad/OneDrive/Desktop/Projects/mortgage-credit-risk/data/03_processed/behavioral/freddie_mac/2015/model-input.parquet")

c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [8]:
model_input.count()

2741047